In [69]:
from nimber_ops import *
from transfinite import Ordinal
import numpy as np

In [3]:
W = Ordinal

In [4]:
# utility functions: collect into util.py later
def base(n : int, b : int, length :  int = 0) -> list[int]:
    ''' returns the base b expansion of n as a  list
    n = sum_j^N a_j*b^j returns [a_0, a_1, ..., a_N]
    optional length parameter to include leading zeros if desired,
    otherwise minimum possible length where highest digit is non-zero'''
    assert (n >= 0 and b > 1), ('positional base expansion defined for' 
                                'non-negative integers and positive bases')
    if length == 0:
        if n == 0: return [0]
        coeffs = []
        while n > 0:
            coeffs.append(n % b)
            n //= b
        return coeffs
    else:
        coeffs = base(n, b)
        while len(coeffs) < length:
            coeffs.append(0)
        return coeffs

def base_eval(coeffs : list[int], b : int) -> int:
    total = 0
    for coeff in coeffs[::-1]:
        total = coeff + b * total
    return total
  
def ord_decomp(ordinal : Ordinal | int) -> list:
    ''' returns [inf_1, inf_2, ..., inf_n, finite term]'''
    if isinstance(ordinal, int):
        assert ordinal >= 0
        return [ordinal]
    high = Ordinal(ordinal.exponent, ordinal.coefficient)
    terms = [high]
    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        terms.append(Ordinal(remainder.exponent, remainder.coefficient, 0))
        remainder = remainder.addend
    assert(isinstance(remainder, int))
    terms.append(remainder)
    return terms

def ord_recomp(list : list) -> Ordinal | int:
    result = 0
    terms = sorted(list, reverse=True)
    for term in list:
        result = result + term
    return result

small_primes = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67]
kappa_p = { # generator of smallest extension of prime degree p
           # kappa_{p_^n} = w^{w^{k-1}*p^{n-1}} where k = |{primes < p}|
           2: 2, 3 : Ordinal()}
for i, prime in enumerate(small_primes[2:]):
    kappa_p[prime] = Ordinal(Ordinal(i+1))
alpha_p = { # (kappa_p)^p, e.g. w^3=2, [w^w]^5 = 4, [w^w^2]^7 = w+1, etc.
           # very hard to calculate for higher p, based on http://www.neverendingbooks.org/on2-extending-lenstras-list/
            2:3, 3:2, 5:4, 7:Ordinal()+1, 11:Ordinal(Ordinal())+1, 13:Ordinal()+4,
            17: 16, 19:Ordinal(3)+4, 23:Ordinal(Ordinal(3))+1, 
            29:Ordinal(Ordinal(2))+4, 31:Ordinal(Ordinal())+1, 37: Ordinal(3)+4,
            41: Ordinal(Ordinal())+1, 43:Ordinal(Ordinal(2))+1, 47:Ordinal(Ordinal(7))+1, 
            53: Ordinal(Ordinal(4))+1, 59:Ordinal(Ordinal(8))+1, 
            61:Ordinal(Ordinal())+Ordinal(), 67:Ordinal(Ordinal(3))+Ordinal()}

In [114]:
class Nim:
    ''' nimbers '''

    def __init__(self, n: int | Ordinal) -> None:
        ''' ordinal considered an a field element in On_2 
        val = ordinal
        field = smallest x > n such that x is a field
        base = largest y < n such that y is a field (or base = 0 for n < 2)
        write n = high * base + low, where low,high < base
        '''
        def exp2(level: int) -> int:
            return 1 << level
        if isinstance(n, int):
            self.val = abs(n)
            self.isfinite = True
            if self.val < 2:
                self.field = 2  # smallest
                self.base = 0
                self.high = 0
                self.low = self.val
            else:
                level = 0
                while n >> (1 << level) > 0:
                    level += 1
                self.field = exp2(exp2(level))
                self.base = exp2(exp2(level - 1))
                self.high = self.val // self.base
                self.low = self.val - self.high * self.base
        else:
            self.val = n
            self.isfinite = False
            # these attributes can be found, but not as useful for infinite nums
            self.field = None
            self.base = None
            self.high=None
            self.low=None 
            # assert isinstance(
            #     n, Ordinal), 'An infinite nimber must be an ordinal'
            # self.val = n
            # self.isfinite = False
            # # find the smallest field containing n
            # if (k := n.exponent) < Ordinal():  # n = omega^k, k fintie => cubic extension
            #     power = 0
            #     while 3 ** power <= k:
            #         power += 1
            #     self.field = Ordinal(3**power)
            #     exp = 3**(power-1)
            #     self.base = Ordinal(exp)
            #     high = Ordinal(n.exponent - exp, n.coefficient,
            #                 0) if exp < n.exponent else n.coefficient
            #     remainder = n.addend
            #     while isinstance(remainder, Ordinal) and remainder.exponent > exp:
            #         high += Ordinal(remainder.exponent - exp,
            #                         remainder.coefficient, 0)
            #         remainder = remainder.addend
            #         if isinstance(remainder, Ordinal) and remainder.exponent == exp:
            #             high += remainder.coefficient
            #             remainder = remainder.addend
            #     self.high = high
            #     self.low = remainder

    def __add__(self, other):
        if self.isfinite and other.isfinite:
            return Nim(self.val ^ other.val)  # finite nim sum is bitwise XOR
        if self.val == other.val:
            return Nim(0)
        ord1, ord2 = self.val, other.val
        terms1, terms2 = ord_decomp(ord1), ord_decomp(ord2)
        sum = {}
        for term in terms1[:-1]:
            sum[term.exponent] = term.coefficient
        for term in terms2[:-1]:  # nim sum the coeffs if any terms with same exp
            try:
                sum[term.exponent] = sum[term.exponent] ^ term.coefficient
            except:
                sum[term.exponent] = term.coefficient

        keys = sorted(sum.keys())  # ordinal addition not commutative
        result = terms1[-1] ^ terms2[-1]  # nim sum of finite part
        for key in keys:
            if sum[key] > 0: # add bigger on the left
                result = Ordinal(key, sum[key]) + result  
        return Nim(result)

    def __eq__(self, other):
        return self.val == other.val

    def __hash__(self):
        return self.val.__hash__()

    def __mul__(self, other):
        assert (isinstance(other, Nim))
        x, y = self.val, other.val
        if x == 0 or y == 0:
            return Nim(0)
        if x == 1:
            return other
        if y == 1:
            return self
        if self.isfinite and other.isfinite:
            if self.val == other.val:
                return self.sq()
            def nim_product(a: int, b: int) -> int:
                # first handle trivial cases
                if a == 0 or b == 0:
                    return 0
                elif a == 1:
                    return b
                elif b == 1:
                    return a
                elif a == 2 and b == 2:
                    return 3
                else:
                    # do euclidean division by greatest possible fermat power
                    # a = q_a * F_a + r_a and b = q_b * F_b + r_b
                    F_a, q_a, r_a = Nim(a).base, Nim(a).high, Nim(a).low
                    F_b, q_b, r_b = Nim(b).base, Nim(b).high, Nim(b).low

                    # if one the Fermat powers is greater than the other, then
                    # nim multiplication by it is the same as ordinary multiplication
                    if F_a < F_b:
                        return nim_product(a, q_b)*F_b ^ nim_product(a, r_b)
                    elif F_a > F_b:
                        return nim_product(q_a, b)*F_a ^ nim_product(r_a, b)
                    else:
                        # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
                        p_1 = nim_product(q_a, q_b)
                        p_2 = nim_product(r_a, r_b)
                        p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
                        p_4 = nim_product(p_1, F_a >> 1)
                        p_5 = p_3 ^ p_2
                        return p_5 * F_a ^ p_2 ^ p_4
            return Nim(nim_product(x, y))
        elif self.isfinite and not other.isfinite:
            terms = ord_decomp(other.val)  # distribute to each coefficient
            terms[-1] = (self * Nim(terms[-1])).val
            for term in terms[:-1]:
                term.coefficient = (self * Nim(term.coefficient)).val
            return Nim(ord_recomp(terms))
        elif not self.isfinite and other.isfinite:
            return other * self
        else:  # both infinite          
            terms1 = ord_decomp(self.val) 
            if self.val == other.val and (len(terms1)>2 or terms1[-1]>0):
                return self.sq()
            terms2 = ord_decomp(other.val)
            inf1, fin1 = terms1[:-1], terms1[-1]
            inf2, fin2 = terms2[:-1], terms2[-1]
            if fin1 == 0 or fin2 == 0:
                to_sum = {Nim(0)}
                #result = Nim(0)
                
            else:
                # start by "FOIL-ing" to handle the terms where one is finite
                to_sum = {Nim(fin1) * other}^{self * Nim(fin2)}^{Nim(fin1) * Nim(fin2)}
                # result = Nim(fin1) * other + self * Nim(fin2) \
                       # + Nim(fin1) * Nim(fin2)
            for x in inf1:  # expand and distribute the purely infinite terms
                for y in inf2:
                    # calculate (w^N * a) x (w^M * b)
                    X, Y = sorted([x, y], reverse=True) # X >= Y
                    N, M = X.exponent, Y.exponent # N >= M
                    a, b = X.coefficient, Y.coefficient
                    coeff = Nim(a) * Nim(b)
                    
                    if N < Ordinal() and M < Ordinal(): # handle finite case
                        N_tern = base(N, 3)  # write exponents in ternary
                        M_tern = base(M, 3)
                        K = max([len(N_tern), len(M_tern)])                        

                        # invert the powers of 3 since w^(3^k) = 2^(3^{-k-1})
                        def phi(n): return base_eval(
                            base(n, 3, K)[::-1], 3)  # reversed 3s
                        exp_p = phi(N) + phi(M)  # the exponent 2^(3^{-K} * exp_2)

                        next_power = 3 ** K
                        q, r = exp_p // next_power, exp_p % next_power
                        coeff = coeff * Nim(2) ** q
                        W = Nim(Ordinal(phi(r))) if r > 0 else Nim(1)
                        to_sum ^= {coeff * W}
                        # result = result + coeff * W
                    elif N < Ordinal() and not M < Ordinal():
                        W = Nim(Ordinal(N + M))
                        to_sum ^= {coeff * W}
                        # result = result + coeff * W
                    else:
                        N_decomp, M_decomp = ord_decomp(N), ord_decomp(M)
                        # multiply all omega terms using exponent rules
                        N_fin_exp, M_fin_exp = N_decomp[-1], M_decomp[-1]
                        N_inf_exp, M_inf_exp = N_decomp[:-1], M_decomp[:-1]
                        # handle finite exponent first
                        omega_N = Ordinal(N_fin_exp) if N_fin_exp!=0 else 1
                        omega_M = Ordinal(M_fin_exp) if M_fin_exp!=0 else 1  
                        # start multiplying omegas                     
                        prod : Ordinal = (Nim(omega_N) * Nim(omega_M)).val
                        # group the exponents by like terms
                        N_dict = {exp.exponent: exp for exp in N_inf_exp}
                        M_dict = {exp.exponent: exp for exp in M_inf_exp}
                        N_exp_exp = set(N_dict)
                        M_exp_exp = set(M_dict)
                        like_terms = N_exp_exp & M_exp_exp
                        unlike_terms = (N_exp_exp | M_exp_exp) - like_terms
                        for t in sorted(unlike_terms): # normal ordinal product
                            mult = N_dict[t] if t in N_dict else M_dict[t]
                            prod = Ordinal(mult) * prod # pull through
                        for n in like_terms:
                            i = N_dict[n].coefficient
                            j = M_dict[n].coefficient
                            p = small_primes[n+1]
                            alpha = Nim(alpha_p[p])
                            i_p = base(i, p)  # write coeff in base p
                            j_p = base(j, p)
                            K = max([len(i_p), len(j_p)])                        

                            # invert the powers of p                                     
                            def phi(n): return base_eval(base(n, p, K)[::-1], p)
                            exp_p = phi(i) + phi(j)  

                            next_power = p ** K
                            q, r = exp_p // next_power, exp_p % next_power
                            coeff = coeff * alpha ** q
                            term = Ordinal(Ordinal(n)*phi(r)) if r > 0 else 1
                            # need recursive because might be new like terms
                            prod = (Nim(prod) * Nim(term)).val
                        to_sum ^= {coeff * Nim(prod)}
                        #result = result + coeff * Nim(prod)
            return Nim(0).sum(*to_sum)

    def __pow__(self, p):
        """
        Compute x**n using exponentiation by squaring.

        """
        if p >= 0:  # binary exponentiation by squaring
            result = Nim(1)
            nimber = self
            while p > 0:
                if p & 1:
                    result = result * nimber
                nimber = nimber.sq()
                p >>= 1
            return result
        elif p == -1:
            if self.field == 2:
                return self
            a, b, F, f = Nim(self.high), Nim(self.low), Nim(
                self.base), Nim(self.base >> 1)
            det = (a + b)*b + a*a*f
            return det**(-1) * (a*F + (a+b))
        else:
            inv = self ** (-1)
            return inv ** (-p)

    def __repr__(self) -> str:
        if self.isfinite:
            return str(self.val)
        else:
            return self.val.__repr__()

    def _repr_latex_(self):
        """
        Special method for Jupyter to render LaTeX.
        """
        if self.isfinite:
            # No special LaTeX for integers, just return the string
            return f"${self.val}$"
        else:
            # Delegate to the Ordinal's LaTeX representation
            return self.val._repr_latex_()

    def deg(self) -> int:
        ''' returns the degree of the minimal polynomial'''
        if self.isfinite:
            return self.field.bit_length() - 1
        else: 
            d = 1
            x = self * self
            while x != self:
                x = x * x
                d += 1
            return d
    def inv(self):
        return self ** (-1)

    def order(self):
        if self.isfinite:
            def fermat_divisors(n: int, include_one: bool = False) -> list:
                '''
                Find the divisors of a Mersenne number 2 ** (2 ** n) - 1
                By default does not include 1
                '''
                # for now, this only works for n < 6
                # could potentiall go up to n = 11 using known factors on wikipedia
                # no one knows the factors of 2 ** (2 ** 11) + 1
                if n >= 6:
                    raise ValueError('This function only works for n < 6')
                else:
                    divisors = []
                    for i in range(0 + int(not include_one), 2 ** n):
                        product = 1
                        for j in range(n):
                            if i >> j & 1:
                                product *= 2 ** (2 ** j) + 1
                        divisors.append(product)
                    return divisors

            n = self.val
            if n == 0:
                return 0
            elif n == 1:
                return 1
            elif n in {2, 3}:
                return 3
            elif n == 3:
                return 3
            elif n < 1 << (1 << 5):
                # make more efficient by only checking possible orders
                # use Lagrange's theorem
                # find the smallest field containing n i.e. smallest F_k > n
                exp = (n.bit_length() - 1).bit_length()
                # find the order of n must divide F_k - 1 which factors by difference of squares
                divisors = fermat_divisors(exp)
                for factor in divisors[:-1]:
                    if (Nim(n)**factor).val == 1:
                        return factor
                else:
                    return divisors[-1]
            else:
                for factor in fermat_divisors(5):
                    if (Nim(n)**factor).val == 1:
                        return factor
                    # brute force: will probably loop forever
                    i = 1 << (1 << 5) + 1
                    while True:
                        if (Nim(i)**factor).val == 1:
                            return i
                        i += 2
        else:
            ...  # inifite case is hard..

    def sum(self, *args):
        if len(args) == 0: return self
        elif len(args) == 1: return self + args[0]
        else:
            *inf, fin = ord_decomp(self.val)
            terms = {term.exponent: term.coefficient for term in inf}
            terms[0] = fin
            for arg in args:
                *inf_, fin_ = ord_decomp(arg.val)
                terms[0] ^= fin_
                for term_ in inf_:
                    try:
                        terms[term_.exponent] = terms[term_.exponent] ^ term_.coefficient
                    except:
                        terms[term_.exponent] = term_.coefficient
            result = terms[0]
            for key in sorted(terms.keys())[1:]:
                if terms[key] > 0:
                    result = Ordinal(key, terms[key]) + result
            return Nim(result)
    
    def sq(self):
        '''returns the Nimber's square using Freshman's Dream'''
        if self.isfinite:
            a, b, base = self.high, self.low, self.base
            # x = a *2^2^n + b
            # x^2 = (a^2)*(2^2^n + 2^(2^n-1)) + b^2
            if self.base == 0: # either 0 or 1
                return self
            else:
                term = base + (base >> 1)
                return Nim(a).sq() * Nim(term) + Nim(b).sq()
        else:
            terms = ord_decomp(self.val)
            sum = Nim(terms[-1]).sq()
            for term in terms[:-1]:
                sum = sum + Nim(term)*Nim(term)
            return sum
                
    def sqrt(self):
        if self.isfinite:
            if self.field == 2:
                return self
            term = self.sq() + self
            return term.sqrt() + self
        else:
            ...  # not sure how to implement...

In [62]:
ww = [Nim(Ordinal(Ordinal(n)*k)*l) \
    for n in range(1, 10) for k in range(1, 10) for l in range (1, 10)]
%timeit ww[0] + ww[1]

6.81 μs ± 80.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [63]:
%%timeit
result = Nim(0)
for w in ww:
    result = result+ w

144 ms ± 324 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [81]:
%timeit sum(ww, start=Nim(0))

632 ms ± 2.87 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [82]:
sum(ww, start=Nim(0)) ==Nim(0).sum(*ww)

True

In [83]:
%timeit Nim(0).sum(*ww)

6.93 ms ± 22.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [68]:
144 * 1000 / (9**3 - 1)

197.8021978021978

In [124]:
a = Nim(Ordinal(Ordinal(6)))
b = a + Nim(1)
a**19

w**3 + 4

In [304]:
# use dynamic programming for the recursive functions
f_p = {}
k_p = {}
k_deg = {}
Q_p = {}
alpha_P = {2:3, 3:2, 5:4}

In [330]:
from Z import is_prime, divs, prime_power, factor, gcd


def lcm(*args):
    if len(args) == 0: return 0
    elif len(args) == 1: return args[0]
    else:
        n, m, *rest = args
    mult = n*m // gcd(n, m)[-1]
    for num in rest:
        mult = mult * num // gcd(mult, num)[-1]
    return mult

def pi(n : int) -> int:
    ''' prime counting function: returns # primes < n'''
    with open('small_primes.txt') as file:
        num_less = 0
        for line in file:
            p = int(line)
            if p < n:
                num_less += 1
            else:
                return num_less
        raise ValueError('n is too big to calculate pi(n) by brute force')
            
            
def f(p):
    '''Lenstra's f function: 
    for prime p, f(p) is min{ n | p divides 2^n-1}
    It is always the case that f(p) divides p-1'''
    assert(is_prime(p)), 'f(p) in only defined for primes p'
    if p in f_p:
        return f_p[p]
    divisors = divs(p-1)
    for div in divisors[:-1]:
        if ((1<<div) -1) % p == 0:
            f_p[p] = div
            return div
    f_p[p] = divisors[-1]
    return divisors[-1]

def kappa(h : int) -> Nim:
    assert h > 0
    if h in k_p:
        return k_p[h]
    if h == 1 : result = Nim(0)
    elif prime_power(h):
        p, n = prime_power(h)
        k = pi(p)
        exp = Ordinal(k, h//p) if k > 0 else h//p
        result = Nim(2**exp)
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = kappa(g)
        else:
            result = kappa(q) + kappa(g)
    k_p[h] = result
    return result
        
def Q(h : int) -> list[int]:
    ''' kappa(h) = sum_{q in Q} kappa(q), q prime powers'''
    assert h > 0
    if h in Q_p:
        return Q_p[h]
    if h == 1 : 
        result = []
    elif prime_power(h):
        result = [h]
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = Q(g)
        else:
            result = Q(g) + Q(q)
    Q_p[h] = result
    return result
def kappa_deg(h : int) -> int:
    ''' faster way to calculate degree for kappa_h'''
    if h in k_deg:
        return k_deg[h]
    if h == 1: result = 1
    
    elif prime_power(h):
        p, n = prime_power(h)
        if p == 2: result = h # deg(k_{2^n}) = 2^n
        else:
            result = h * lcm(kappa_deg(f(p)), (alpha(p) + kappa(f(p))).deg())
    else:
        result = lcm(*[kappa_deg(q) for q in Q(h)])
    k_deg[h] = result
    return result
    
def alpha(p : int) -> Nim:
    assert is_prime(p)
    if p in alpha_p:
        return Nim(alpha_p[p])
    else:
        d = kappa_deg(f(p))
        mersenne = ((1<<d) - 1)
        excess = 0
        # save some iterations with known lower bound for excess
        if len(Q(f(p))) == 1 and Q(f(p))[0] % 2 == 1:
            excess = 1 # Q(f(p)) = {q} odd prime power ==> excess >= 1
        if f(p) % 2 == 0 and prime_power(f(p)//2):
                if prime_power(f(p)//2)[0] == 3:
                    excess = 4 #f(p) = 2*3^k, k>0 ==> excess >=4           
        beta = Nim(kappa(f(p)).val + excess)
        expr = (mersenne % p > 0) or beta**(mersenne//p) == Nim(1)
        while expr:
            excess += 1
            beta = Nim(kappa(f(p)).val + excess)
            d = lcm(kappa_deg(f(p)), Nim(excess).deg())
            mersenne = ((1<<d) - 1)
            expr = (mersenne % p > 0) or beta**(mersenne//p) == Nim(1)
        alpha_p[p] = beta.val
        return beta
    

In [306]:
import time


In [307]:
import pandas as pd

In [ ]:
Nim(W(W(7)))**10000

In [308]:
d = {'p':alpha_P.keys(), 
     'kappa_p':[str(kappa(p)) for p in alpha_P], 
     'alpha_P': [str(alpha_P[p]) for p in alpha_P], 
     'excess' : [str(alpha(p)+kappa(f(p))) for p in alpha_P],
     'f(p)':[f(p) for p in alpha_P],
     'Q(f(p))': [Q(f(p)) for p in alpha_P], 
     'compute_sec':[0., 0., 0.]}
df = pd.DataFrame(d)
df

,p,kappa_p,alpha_P,excess,f(p),Q(f(p)),compute_sec
0,2,2,3,3,1,[],0.0
1,3,w,2,0,2,[2],0.0
2,5,w**w,4,0,4,[4],0.0


In [ ]:
# let's compute higher alphas

# with open('small_primes.txt') as file:
#     for line in file:
        
#         p = int(line)
        
#         start = time.time()
#         alpha(p)
#         end = time.time()
#         alphaP = alpha_p[p]
#         df_p = pd.DataFrame({'p':p, 
#                             'kappa_p':[str(kappa(p))], 
#                             'alpha_P': [str(alpha_P[p])], 
#                             'excess' : [str(alpha(p)+kappa(f(p)))],
#                             'f(p)':[f(p)],
#                             'Q(f(p))': [Q(f(p))], 
#                             'compute_sec':end - start})
#         df = pd.concat([df, df_p], ignore_index=True)
#         df.to_csv('lenstra_nimber.csv', index=False)
#         print(f'alpha_{p}={alphaP}, was computed in {end - start}')
        
        # 47 takes too long :( my code for multiplication is too inefficient

In [ ]:
excess = [3,0,0,1,1,0,0,4,1,0,1,0,1,1,1,1,1,0,0,0,1,1,1,1,0,
 1,0,1,0,0,1,0,1,0,1,0,1,4,1,0,1,0,0,0,0,0,0,1,1,1,
 1,0,0,1,0,1,1,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,1,0,0,
 1,1,1,0,0,0,0,1,0,1,0,0,1,1,0,1,1,1,0,1,1,0,0,0,0,
 0,1,1,1,1,0] # use https://oeis.org/A380496/